In [153]:
import pandas as pd
import math
from scipy.stats import levene, shapiro, kruskal
from statsmodels.formula.api import ols
from scikit_posthocs import posthoc_dunn
import statsmodels.formula.api as smf

In [155]:
# define file paths
file = path

In [318]:
# import files
total_data = pd.read_excel(file+'/total_data.xlsx')
county_data = pd.read_excel(file+'/county_data.xlsx')

In [319]:
# data check
total_data

,region_code,state,region,urban_type,time,observed_deaths,expected_deaths,excess_deaths,excess_deaths_per_100k,CumulativeExcessDeaths
0,US1129,Alabama,"Washington County, AL",Shrinking,2020/03,23,16.468087,6.531913,42.617035,42.617035
1,US1129,Alabama,"Washington County, AL",Shrinking,2020/04,11,16.952484,-5.952484,-38.836590,3.780444
2,US1129,Alabama,"Washington County, AL",Shrinking,2020/05,25,13.468087,11.531913,75.239205,79.019649
3,US1129,Alabama,"Washington County, AL",Shrinking,2020/06,16,13.152484,2.847516,18.578429,97.598077
4,US1129,Alabama,"Washington County, AL",Shrinking,2020/07,24,12.868087,11.131913,72.629431,170.227508
...,...,...,...,...,...,...,...,...,...,...
41107,US39147,Ohio,"Seneca County, OH","Pop-Decline, Econ-Growth",2022/10,46,51.424362,-5.424362,-9.928910,484.752714
41108,US39147,Ohio,"Seneca County, OH","Pop-Decline, Econ-Growth",2022/11,43,49.356883,-6.356883,-11.635824,473.116890
41109,US39147,Ohio,"Seneca County, OH","Pop-Decline, Econ-Growth",2022/12,67,63.224362,3.775638,6.911038,480.027928
41110,US39147,Ohio,"Seneca County, OH","Pop-Decline, Econ-Growth",2023/01,67,67.226798,-0.226798,-0.415138,479.612790


In [320]:
# data check
county_data

,region_code,state,region,type,observed_deaths,expected_deaths,excess_deaths,excess_deaths_per_100k,peak,Classification,...,GRDP_CAGR,Income,r_older,Education,Unemployment,r_white,r_black,r_AmericanIndianandAlaskaNative,r_Asian,r_NativeHawaiianandOtherPacificIslander
0,US55123,Wisconsin,"Vernon County, WI","Pop-Growth, Econ-Decline",30.611111,23.832633,6.778478,21.932669,7,4,...,-0.358675,60.041,19.525743,23.512555,2.7,96.674144,0.243831,0.149550,0.338112,0.035762
1,US48209,Texas,"Hays County, TX",Growing,130.194444,107.668185,22.526259,8.672325,3,2,...,5.164925,79.336,10.988587,38.673646,3.0,82.658744,4.131007,0.607646,1.473340,0.048468
2,US1005,Alabama,"Barbour County, AL",Shrinking,30.000000,29.687519,0.312481,1.236664,8,5,...,-1.312064,38.649,19.420441,11.153098,4.1,46.299848,47.666427,0.351634,0.487493,0.003996
3,US48203,Texas,"Harrison County, TX","Pop-Growth, Econ-Decline",69.472222,55.065716,14.406507,20.758928,11,3,...,-2.215375,56.645,16.900459,21.124477,4.5,71.577982,20.708672,0.344118,0.812959,0.034562
4,US48091,Texas,"Comal County, TX",Growing,140.805556,125.691813,15.113743,8.750407,8,2,...,4.948369,93.487,18.082131,39.601859,3.5,87.100543,2.172964,0.252483,1.167733,0.043647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,US1049,Alabama,"DeKalb County, AL",Growing,86.111111,72.096729,14.014382,19.519143,7,5,...,0.361412,45.062,17.261329,14.432755,2.3,84.773905,1.530169,1.189976,0.106398,0.128797
1138,US47167,Tennessee,"Tipton County, TN",Growing,65.027778,50.983483,14.044294,22.912860,8,2,...,1.274046,62.814,14.459005,18.675892,3.6,76.974432,18.542283,0.245281,0.700107,0.000000
1139,US55043,Wisconsin,"Grant County, WI",Growing,46.638889,43.563523,3.075366,5.942116,9,5,...,1.508574,57.861,17.253753,23.858838,2.7,96.164437,1.555168,0.135738,0.971495,0.000000
1140,US18113,Indiana,"Noble County, IN",Growing,44.250000,44.677664,-0.427664,-0.905889,7,5,...,1.289240,65.979,16.041824,16.311432,2.8,94.848866,0.631822,0.054576,0.686398,0.069270


## 1. Descriptive statistics

#### (1) All Counties

In [198]:
total_data.sum()

region_code               US1129US1129US1129US1129US1129US1129US1129US11...
state                     AlabamaAlabamaAlabamaAlabamaAlabamaAlabamaAlab...
region                    Washington County, ALWashington County, ALWash...
urban_type                ShrinkingShrinkingShrinkingShrinkingShrinkingS...
time                      2020/032020/042020/052020/062020/072020/082020...
observed_deaths                                                     6984343
expected_deaths                                              6054965.494768
excess_deaths                                                 929377.505232
excess_deaths_per_100k                                        575729.275872
CumulativeExcessDeaths                                      11609648.553874
dtype: object

In [200]:
# Mean, Standard Deviation, Minimum, and Maximum, standard error and 95% confidence interval for Supplementary Table 1
temp_1 = total_data.describe().T
temp_1 = temp_1.drop(['25%', '50%', '75%'], axis = 1)
temp_1['count'] = temp_1['count'].astype(int)
temp_1['ste'] = temp_1['std'].astype(float)/math.sqrt(1142)
temp_1['low_ci'] = temp_1['mean'] - 1.96*temp_1['ste']
temp_1['hi_ci'] = temp_1['mean'] + 1.96*temp_1['ste']
temp_1.round(2)

,count,mean,std,min,max,ste,low_ci,hi_ci
observed_deaths,41112,169.89,352.27,10.00,14778.00,10.42,149.45,190.32
expected_deaths,41112,147.28,289.31,9.58,6547.87,8.56,130.50,164.06
excess_deaths,41112,22.61,97.21,-124.25,8319.90,2.88,16.97,28.24
excess_deaths_per_100k,41112,14.00,24.39,-101.91,281.70,0.72,12.59,15.42
CumulativeExcessDeaths,41112,282.39,257.07,-580.82,2177.61,7.61,267.48,297.30


In [202]:
county_data.sum()

region_code                                US55123US48209US1005US48203US48091US19193US101...
state                                      WisconsinTexasAlabamaTexasTexasIowaAlabamaNew ...
region                                     Vernon County, WIHays County, TXBarbour County...
type                                       Pop-Growth, Econ-DeclineGrowingShrinkingPop-Gr...
observed_deaths                                                                194009.527778
expected_deaths                                                                168193.485966
excess_deaths                                                                   25816.041812
excess_deaths_per_100k                                                          15992.479885
peak                                                                                    7776
Classification                                                                          4431
pop                                                                   

In [329]:
# Mean, Standard Deviation, Minimum, and Maximum, standard error and 95% confidence interval for Table 3
temp_1 = county_data.describe().T
temp_1 = temp_1.drop(['25%', '50%', '75%'], axis = 1)
temp_1['count'] = temp_1['count'].astype(int)
temp_1['ste'] = temp_1['std'].astype(float)/math.sqrt(1142)
temp_1['low_ci'] = temp_1['mean'] - 1.96*temp_1['ste']
temp_1['hi_ci'] = temp_1['mean'] + 1.96*temp_1['ste']
temp_1.round(2)


temp_1 = temp_1.round(2)
pd.options.display.float_format = '{:.2f}'.format

temp_1 

,count,mean,std,min,max,ste,low_ci,hi_ci
observed_deaths,1142,169.89,341.58,15.56,6667.33,10.11,150.07,189.70
expected_deaths,1142,147.28,288.64,12.70,5481.40,8.54,130.54,164.02
excess_deaths,1142,22.61,54.92,-3.09,1185.94,1.63,19.42,25.79
excess_deaths_per_100k,1142,14.00,8.53,-16.13,60.49,0.25,13.51,14.50
peak,1142,6.81,2.64,0.00,13.00,0.08,6.66,6.96
Classification,1142,3.88,1.54,1.00,6.00,0.05,3.79,3.97
"total_Area (Land, in square meters)",1142,2678419900.42,3766591469.64,67606554.00,52072729527.00,111459083.22,2459960097.30,2896879703.53
"total_Area (Land, in square kilo meters)",1142,2678.42,3766.59,67.61,52072.73,111.46,2459.96,2896.88
pop,1142,209963.30,488943.33,7222.00,9833540.33,14468.57,181604.91,238321.69
PopDensity,1142,157.32,743.18,1.16,18320.17,21.99,114.21,200.42


#### (2) Each urban types

A. Growing cities

In [331]:
# total_deaths, expected_deaths,excess_deaths, excess_deaths per 100k, number of peak, standard error and 95% confidence interval 
temp_growing = county_data[county_data['type']=='Growing'].describe().T
temp_growing = temp_growing.drop(['25%', '50%', '75%'], axis = 1)
temp_growing['count'] = temp_growing['count'].astype(int)
temp_growing['ste'] = temp_growing['std'].astype(float)/math.sqrt(692)
temp_growing['low_ci'] = temp_growing['mean'] - 1.96*temp_growing['ste']
temp_growing['hi_ci'] = temp_growing['mean'] + 1.96*temp_growing['ste']
temp_growing.round(2)

,count,mean,std,min,max,ste,low_ci,hi_ci
observed_deaths,691,233.01,416.26,15.56,6667.33,15.82,201.99,264.02
expected_deaths,691,202.06,350.55,12.70,5481.40,13.33,175.94,228.18
excess_deaths,691,30.95,68.12,-1.26,1185.94,2.59,25.87,36.02
excess_deaths_per_100k,691,11.62,6.55,-5.69,42.17,0.25,11.13,12.10
peak,691,5.87,2.65,0.00,13.00,0.10,5.67,6.07
Classification,691,3.31,1.43,1.00,6.00,0.05,3.20,3.42
"total_Area (Land, in square meters)",691,2823617025.29,3796116820.43,67606554.00,52072729527.00,144306708.70,2540775876.23,3106458174.35
"total_Area (Land, in square kilo meters)",691,2823.62,3796.12,67.61,52072.73,144.31,2540.78,3106.46
pop,691,304530.70,603009.43,12745.00,9833540.33,22922.98,259601.66,349459.74
PopDensity,691,227.52,940.30,2.10,18320.17,35.74,157.46,297.58


B. Sririnking cities

In [334]:
# total_deaths, expected_deaths,excess_deaths, excess_deaths per 100k, number of peak, standard error and 95% confidence interval 
temp_shrinking = county_data[county_data['type']=='Shrinking'].describe().T
temp_shrinking = temp_shrinking.drop(['25%', '50%', '75%'], axis = 1)
temp_shrinking['count'] = temp_shrinking['count'].astype(int)
temp_shrinking['ste'] = temp_shrinking['std'].astype(float)/math.sqrt(241)
temp_shrinking['low_ci'] = temp_shrinking['mean'] - 1.96*temp_shrinking['ste']
temp_shrinking['hi_ci'] = temp_shrinking['mean'] + 1.96*temp_shrinking['ste']
temp_shrinking.round(2)

,count,mean,std,min,max,ste,low_ci,hi_ci
observed_deaths,243,63.39,55.23,17.58,301.17,3.56,56.42,70.36
expected_deaths,243,54.43,48.11,13.46,254.72,3.10,48.36,60.50
excess_deaths,243,8.96,8.13,-0.80,53.10,0.52,7.94,9.99
excess_deaths_per_100k,243,19.11,9.87,-4.91,60.49,0.64,17.86,20.35
peak,243,8.39,1.76,3.00,13.00,0.11,8.17,8.61
Classification,243,4.87,1.19,1.00,6.00,0.08,4.72,5.02
"total_Area (Land, in square meters)",243,2513302129.10,3514158591.70,171388147.00,29057636965.00,226366854.31,2069623094.66,2956981163.54
"total_Area (Land, in square kilo meters)",243,2513.30,3514.16,171.39,29057.64,226.37,2069.62,2956.98
pop,243,53698.89,52812.65,10801.33,310408.67,3401.96,47031.04,60366.74
PopDensity,243,41.48,128.26,1.16,1709.97,8.26,25.28,57.67


C. Pop-Growth, Econ-Decline

In [337]:
# total_deaths, expected_deaths,excess_deaths, excess_deaths per 100k, number of peak, standard error and 95% confidence interval 
temp_PGED = county_data[county_data['type']=='Pop-Growth, Econ-Decline'].describe().T
temp_PGED = temp_PGED.drop(['25%', '50%', '75%'], axis = 1)
temp_PGED['count'] = temp_PGED['count'].astype(int)
temp_PGED['ste'] = temp_PGED['std'].astype(float)/math.sqrt(36)
temp_PGED['low_ci'] = temp_PGED['mean'] - 1.96*temp_PGED['ste']
temp_PGED['hi_ci'] = temp_PGED['mean'] + 1.96*temp_PGED['ste']
temp_PGED.round(2)

,count,mean,std,min,max,ste,low_ci,hi_ci
observed_deaths,39,94.58,67.20,24.72,279.31,11.20,72.63,116.53
expected_deaths,39,81.64,59.15,17.17,235.46,9.86,62.31,100.96
excess_deaths,39,12.95,9.05,1.31,43.85,1.51,9.99,15.90
excess_deaths_per_100k,39,15.23,8.81,4.80,43.49,1.47,12.35,18.10
peak,39,6.87,1.85,4.00,11.00,0.31,6.27,7.48
Classification,39,3.85,1.25,2.00,6.00,0.21,3.44,4.25
"total_Area (Land, in square meters)",39,4960030383.92,8235422426.49,516760664.00,44553803874.00,1372570404.41,2269792391.27,7650268376.58
"total_Area (Land, in square kilo meters)",39,4960.03,8235.42,516.76,44553.80,1372.57,2269.79,7650.27
pop,39,103186.45,81532.85,22656.33,336696.00,13588.81,76552.39,129820.52
PopDensity,39,72.29,116.21,1.21,651.55,19.37,34.33,110.25


D. Pop-Decline, Econ-Growth

In [340]:
# total_deaths, expected_deaths,excess_deaths, excess_deaths per 100k, number of peak, standard error and 95% confidence interval 
temp_PDEG = county_data[county_data['type']=='Pop-Decline, Econ-Growth'].describe().T
temp_PDEG = temp_PDEG.drop(['25%', '50%', '75%'], axis = 1)
temp_PDEG['count'] = temp_PDEG['count'].astype(int)
temp_PDEG['ste'] = temp_PDEG['std'].astype(float)/math.sqrt(173)
temp_PDEG['low_ci'] = temp_PDEG['mean'] - 1.96*temp_PDEG['ste']
temp_PDEG['hi_ci'] = temp_PDEG['mean'] + 1.96*temp_PDEG['ste']
temp_PDEG.round(2)

,count,mean,std,min,max,ste,low_ci,hi_ci
observed_deaths,169,82.30,183.08,16.69,1802.42,13.92,55.02,109.58
expected_deaths,169,71.96,159.57,14.04,1533.16,12.13,48.18,95.74
excess_deaths,169,10.34,24.19,-3.09,269.25,1.84,6.74,13.95
excess_deaths_per_100k,169,16.14,9.68,-16.13,43.39,0.74,14.70,17.59
peak,169,8.36,1.89,2.00,13.00,0.14,8.08,8.64
Classification,169,4.79,1.30,1.00,6.00,0.10,4.59,4.98
"total_Area (Land, in square meters)",169,1795636446.49,1385866417.62,238408071.00,16441083046.00,105365472.61,1589120120.17,2002152772.80
"total_Area (Land, in square kilo meters)",169,1795.64,1385.87,238.41,16441.08,105.37,1589.12,2002.15
pop,169,72628.81,175979.25,7222.00,1764997.33,13379.45,46405.08,98852.54
PopDensity,169,56.45,203.44,2.66,2420.98,15.47,26.13,86.77


#### (3) Each Urban-Rural Classification

In [300]:
county_data.groupby(['Classification'])[['excess_deaths_per_100k','peak']].describe().round(2)

excess_deaths_per_100k                                     \
                                count   mean    std    min    25%    50%   
Classification                                                             
1                                56.0   9.91   3.20   3.75   7.24   9.73   
2                               227.0  10.96   5.93  -0.16   6.92   9.49   
3                               207.0  12.25   6.32   0.38   7.84  11.45   
4                               159.0  12.55   7.50 -16.13   7.14  11.70   
5                               294.0  15.09   8.66  -5.78   9.12  14.32   
6                               199.0  20.01  10.97  -4.91  11.82  19.06   

                               peak                                         
                  75%    max  count  mean   std  min  25%  50%   75%   max  
Classification                                                              
1               12.46  17.10   56.0  2.84  1.30  0.0  2.0  3.0   4.0   6.0  
2               13.71  34.18  227.0  5.26  2.51  0.0  3.0  5.0   7.0  12.0  
3               15.29  38.20  207.0  5.62  2.51  0.0  4.0  5.0   7.0  13.0  
4               17.58  34.74  159.0  6.62  1.90  1.0  5.0  7.0   8.0  11.0  
5               20.48  42.17  294.0  8.12  1.58  2.0  7.0  8.0   9.0  12.0  
6               27.14  60.49  199.0  9.16  1.54  6.0  8.0  9.0  10.0  13.0

In [304]:
# total_deaths, expected_deaths,excess_deaths, excess_deaths per 100k, number of peak, standard error and 95% confidence interval 
group_stats = county_data.groupby('Classification')[['excess_deaths_per_100k', 'peak']].agg(
    count=('excess_deaths_per_100k', 'count'),
    mean_excess=('excess_deaths_per_100k', 'mean'),
    std_excess=('excess_deaths_per_100k', 'std'),
    mean_peak=('peak', 'mean'),
    std_peak=('peak', 'std')
)

# Standard Error (SE)
group_stats['se_excess'] = group_stats['std_excess'] / np.sqrt(group_stats['count'])
group_stats['se_peak'] = group_stats['std_peak'] / np.sqrt(group_stats['count'])

# 95% Confidence Interval
z = 1.96
group_stats['ci_lower_excess'] = group_stats['mean_excess'] - z * group_stats['se_excess']
group_stats['ci_upper_excess'] = group_stats['mean_excess'] + z * group_stats['se_excess']

group_stats['ci_lower_peak'] = group_stats['mean_peak'] - z * group_stats['se_peak']
group_stats['ci_upper_peak'] = group_stats['mean_peak'] + z * group_stats['se_peak']

# 반올림
group_stats = group_stats.round(2)

group_stats

,count,mean_excess,std_excess,mean_peak,std_peak,se_excess,se_peak,ci_lower_excess,ci_upper_excess,ci_lower_peak,ci_upper_peak
Classification,,,,,,,,,,,
1,56,9.91,3.20,2.84,1.30,0.43,0.17,9.07,10.75,2.50,3.18
2,227,10.96,5.93,5.26,2.51,0.39,0.17,10.19,11.74,4.93,5.58
3,207,12.25,6.32,5.62,2.51,0.44,0.17,11.39,13.11,5.28,5.96
4,159,12.55,7.50,6.62,1.90,0.60,0.15,11.38,13.72,6.32,6.91
5,294,15.09,8.66,8.12,1.58,0.51,0.09,14.10,16.08,7.93,8.30
6,199,20.01,10.97,9.16,1.54,0.78,0.11,18.49,21.53,8.95,9.38


In [312]:
county_data.groupby(['type','Classification']).describe()

observed_deaths               \
                                                  count         mean   
type                     Classification                                
Growing                  1                         52.0  1114.072115   
                         2                        194.0   231.408219   
                         3                        162.0   229.901235   
                         4                        102.0   115.694989   
                         5                        132.0    55.902778   
                         6                         49.0    35.887755   
Pop-Decline, Econ-Growth 1                          3.0  1243.333333   
                         2                         12.0    99.344907   
                         3                         14.0   161.698413   
                         4                         19.0    93.021930   
                         5                         62.0    50.235215   
                         6                         59.0    31.209510   
Pop-Growth, Econ-Decline 2                          6.0    98.157407   
                         3                         11.0   148.181818   
                         4                          9.0    98.416667   
                         5                          9.0    52.388889   
                         6                          4.0    28.125000   
Shrinking                1                          1.0   287.805556   
                         2                         15.0   102.135185   
                         3                         20.0   139.047222   
                         4                         29.0   100.188697   
                         5                         91.0    53.558913   
                         6                         87.0    34.757982   

                                                                              \
                                                 std         min         25%   
type                     Classification                                        
Growing                  1               1026.381645  186.111111  562.243056   
                         2                258.002003   17.583333   74.923611   
                         3                169.391078   15.555556  107.125000   
                         4                 65.913337   20.666667   70.097222   
                         5                 27.885344   16.888889   35.361111   
                         6                 14.625964   16.833333   24.138889   
Pop-Decline, Econ-Growth 1                591.785556  623.527778  963.791667   
                         2                115.923148   16.694444   28.680556   
                         3                161.639673   31.000000   47.145833   
                         4                 53.453602   19.305556   48.527778   
                         5                 21.001745   20.555556   37.486111   
                         6                 12.364516   17.083333   21.263889   
Pop-Growth, Econ-Decline 2                 75.852038   40.138889   44.861111   
                         3                 73.175829   52.777778   84.916667   
                         4                 50.885170   30.611111   62.861111   
                         5                 19.571923   26.500000   37.277778   
                         6                  2.795637   24.722222   27.097222   
Shrinking                1                       NaN  287.805556  287.805556   
                         2                 65.755758   31.916667   46.972222   
                         3                100.214583   23.361111   57.041667   
                         4                 58.051901   20.333333   50.555556   
                         5                 27.580382   20.333333   33.847222   
                         6                 14.190350   17.583333   23.472222   

                                                                   \
                 

## 2. Kruskal-Wallis test

#### (1) Monthly excss death per 100,000 people

A. Test for levene, shapiro

In [93]:
county_data

,region_code,state,region,type,observed_deaths,expected_deaths,excess_deaths,excess_deaths_per_100k,peak,POP_CAGR,GRDP_CAGR,Income,The_elderly,Education,Unemployment,Race_white,Race_black
0,US55123,Wisconsin,"Vernon County, WI","Pop-Growth, Econ-Decline",30.611111,23.832633,6.778478,21.932669,7,0.333860,-0.358675,60.041,19.525743,23.512555,2.7,96.674144,0.243831
1,US48209,Texas,"Hays County, TX",Growing,130.194444,107.668185,22.526259,8.672325,3,4.378620,5.164925,79.336,10.988587,38.673646,3.0,82.658744,4.131007
2,US1005,Alabama,"Barbour County, AL",Shrinking,30.000000,29.687519,0.312481,1.236664,8,-0.974886,-1.312064,38.649,19.420441,11.153098,4.1,46.299848,47.666427
3,US48203,Texas,"Harrison County, TX","Pop-Growth, Econ-Decline",69.472222,55.065716,14.406507,20.758928,11,0.273632,-2.215375,56.645,16.900459,21.124477,4.5,71.577982,20.708672
4,US48091,Texas,"Comal County, TX",Growing,140.805556,125.691813,15.113743,8.750407,8,4.154444,4.948369,93.487,18.082131,39.601859,3.5,87.100543,2.172964
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,US1049,Alabama,"DeKalb County, AL",Growing,86.111111,72.096729,14.014382,19.519143,7,0.072863,0.361412,45.062,17.261329,14.432755,2.3,84.773905,1.530169
1138,US47167,Tennessee,"Tipton County, TN",Growing,65.027778,50.983483,14.044294,22.912860,8,0.067426,1.274046,62.814,14.459005,18.675892,3.6,76.974432,18.542283
1139,US55043,Wisconsin,"Grant County, WI",Growing,46.638889,43.563523,3.075366,5.942116,9,0.052381,1.508574,57.861,17.253753,23.858838,2.7,96.164437,1.555168
1140,US18113,Indiana,"Noble County, IN",Growing,44.250000,44.677664,-0.427664,-0.905889,7,0.039546,1.289240,65.979,16.041824,16.311432,2.8,94.848866,0.631822


In [44]:
from scipy.stats import shapiro

# Perform Shapiro-Wilk test for normality for each type
results = {}

for type_name, group in county_data.groupby('type'):
    stat, p_value = shapiro(group['excess_deaths_per_100k'])
    results[type_name] = (stat, p_value)

# Output the results
print("Shapiro-Wilk Test Results by Type:")

for type_name, (stat, p_value) in results.items():
    print(f"\nType: {type_name}")
    print(f"Test Statistic: {stat}")
    print(f"P-value: {p_value}")
    if p_value > 0.05:
        print("Sample looks Gaussian (fail to reject H0)")
    else:
        print("Sample does not look Gaussian (reject H0)")

Shapiro-Wilk Test Results by Type:

Type: Growing
Test Statistic: 0.95488440990448
P-value: 1.0296703946291799e-13
Sample does not look Gaussian (reject H0)

Type: Pop-Decline, Econ-Growth
Test Statistic: 0.9837412238121033
P-value: 0.04568155109882355
Sample does not look Gaussian (reject H0)

Type: Pop-Growth, Econ-Decline
Test Statistic: 0.8945446014404297
P-value: 0.0015448625199496746
Sample does not look Gaussian (reject H0)

Type: Shrinking
Test Statistic: 0.9814624190330505
P-value: 0.0028974839951843023
Sample does not look Gaussian (reject H0)


In [46]:
from scipy.stats import levene

# Prepare data for Levene's test
groups = [group['excess_deaths_per_100k'].values for name, group in county_data.groupby('type')]

# Perform Levene's test for equal variances
levene_stat, levene_p_value = levene(*groups)

# Output the Levene's test results
print("\nLevene's Test for Equal Variances:")
print(f"Test Statistic: {levene_stat}")
print(f"P-value: {levene_p_value}")
if levene_p_value > 0.05:
    print("Variances are equal (fail to reject H0)")
else:
    print("Variances are not equal (reject H0)")


Levene's Test for Equal Variances:
Test Statistic: 23.278318510181393
P-value: 1.2590298785291508e-14
Variances are not equal (reject H0)


B. test for kruska-wallis and posthoc

In [48]:
# Perform Kruskal-Wallis test
kruskal_test = levene(county_data['excess_deaths_per_100k'][county_data['type'] == 'Growing'],
                     county_data['excess_deaths_per_100k'][county_data['type'] == 'Shrinking'],
                     county_data['excess_deaths_per_100k'][county_data['type'] == 'Pop-Growth, Econ-Decline'],
                     county_data['excess_deaths_per_100k'][county_data['type'] == 'Pop-Decline, Econ-Growth'])

print(f"Kruskal-Wallis test statistic: {kruskal_test.statistic:.4f}, p-value: {kruskal_test.pvalue:.4f}")

# Perform Dunn's post-hoc test with Bonferroni correction
posthoc_results = posthoc_dunn(county_data, val_col='excess_deaths_per_100k', group_col='type', p_adjust='bonferroni')

# Print post-hoc results
print(posthoc_results)

# Convert posthoc_results to a styled HTML table for better visualization (if needed)
styled_posthoc_results = posthoc_results.style.background_gradient(cmap='viridis').set_caption("Dunn's Post-hoc Test Results")

# Display the styled table in a Jupyter Notebook
styled_posthoc_results

Kruskal-Wallis test statistic: 23.2783, p-value: 0.0000
                               Growing  Pop-Decline, Econ-Growth  \
Growing                   1.000000e+00              1.239788e-09   
Pop-Decline, Econ-Growth  1.239788e-09              1.000000e+00   
Pop-Growth, Econ-Decline  1.058170e-01              1.000000e+00   
Shrinking                 4.767144e-29              1.614971e-02   

                          Pop-Growth, Econ-Decline     Shrinking  
Growing                                   0.105817  4.767144e-29  
Pop-Decline, Econ-Growth                  1.000000  1.614971e-02  
Pop-Growth, Econ-Decline                  1.000000  4.969007e-02  
Shrinking                                 0.049690  1.000000e+00  


,Growing,"Pop-Decline, Econ-Growth","Pop-Growth, Econ-Decline",Shrinking
Growing,1.000000,0.000000,0.105817,0.000000
"Pop-Decline, Econ-Growth",0.000000,1.000000,1.000000,0.016150
"Pop-Growth, Econ-Decline",0.105817,1.000000,1.000000,0.049690
Shrinking,0.000000,0.016150,0.049690,1.000000


#### (2) Number of peaks

A. Test for levene, shapiro

In [42]:
from scipy.stats import shapiro

# Perform Shapiro-Wilk test for normality for each type
results = {}

for type_name, group in county_data.groupby('type'):
    stat, p_value = shapiro(group['peak'])
    results[type_name] = (stat, p_value)

# Output the results
print("Shapiro-Wilk Test Results by Type:")

for type_name, (stat, p_value) in results.items():
    print(f"\nType: {type_name}")
    print(f"Test Statistic: {stat}")
    print(f"P-value: {p_value}")
    if p_value > 0.05:
        print("Sample looks Gaussian (fail to reject H0)")
    else:
        print("Sample does not look Gaussian (reject H0)")

Shapiro-Wilk Test Results by Type:

Type: Growing
Test Statistic: 0.9755408763885498
P-value: 2.427344769628803e-09
Sample does not look Gaussian (reject H0)

Type: Pop-Decline, Econ-Growth
Test Statistic: 0.9634349346160889
P-value: 0.00020233051327522844
Sample does not look Gaussian (reject H0)

Type: Pop-Growth, Econ-Decline
Test Statistic: 0.9525003433227539
P-value: 0.09978008270263672
Sample looks Gaussian (fail to reject H0)

Type: Shrinking
Test Statistic: 0.9631699323654175
P-value: 6.678373210888822e-06
Sample does not look Gaussian (reject H0)


In [99]:
from scipy.stats import levene

# Prepare data for Levene's test
groups = [group['peak'].values for name, group in county_data.groupby('type')]

# Perform Levene's test for equal variances
levene_stat, levene_p_value = levene(*groups)

# Output the Levene's test results
print("\nLevene's Test for Equal Variances:")
print(f"Test Statistic: {levene_stat}")
print(f"P-value: {levene_p_value}")
if levene_p_value > 0.05:
    print("Variances are equal (fail to reject H0)")
else:
    print("Variances are not equal (reject H0)")


Levene's Test for Equal Variances:
Test Statistic: 30.47013223049633
P-value: 5.998402302997084e-19
Variances are not equal (reject H0)


B. test for kruska-wallis and posthoc

In [100]:
# Perform Kruskal-Wallis test
kruskal_test = levene(county_data['peak'][county_data['type'] == 'Growing'],
                     county_data['peak'][county_data['type'] == 'Shrinking'],
                     county_data['peak'][county_data['type'] == 'Pop-Growth, Econ-Decline'],
                     county_data['peak'][county_data['type'] == 'Pop-Decline, Econ-Growth'])

print(f"Kruskal-Wallis test statistic: {kruskal_test.statistic:.4f}, p-value: {kruskal_test.pvalue:.4f}")

# Perform Dunn's post-hoc test with Bonferroni correction
posthoc_results = posthoc_dunn(county_data, val_col='peak', group_col='type', p_adjust='bonferroni')

# Print post-hoc results
print(posthoc_results.to_string())

# Convert posthoc_results to a styled HTML table for better visualization (if needed)
styled_posthoc_results = posthoc_results.style.background_gradient(cmap='viridis').set_caption("Dunn's Post-hoc Test Results")

# Display the styled table in a Jupyter Notebook
styled_posthoc_results

Kruskal-Wallis test statistic: 30.4701, p-value: 0.0000
                               Growing  Pop-Decline, Econ-Growth  Pop-Growth, Econ-Decline     Shrinking
Growing                   1.000000e+00              7.272355e-28                  0.344657  2.425303e-37
Pop-Decline, Econ-Growth  7.272355e-28              1.000000e+00                  0.001885  1.000000e+00
Pop-Growth, Econ-Decline  3.446573e-01              1.884961e-03                  1.000000  9.864416e-04
Shrinking                 2.425303e-37              1.000000e+00                  0.000986  1.000000e+00


,Growing,"Pop-Decline, Econ-Growth","Pop-Growth, Econ-Decline",Shrinking
Growing,1.000000,0.000000,0.344657,0.000000
"Pop-Decline, Econ-Growth",0.000000,1.000000,0.001885,1.000000
"Pop-Growth, Econ-Decline",0.344657,0.001885,1.000000,0.000986
Shrinking,0.000000,1.000000,0.000986,1.000000


## 3. A mixed effect model

#### (1) Monthly excss death per 100,000 people

A. null model

In [342]:
# Fit a null mixed-effects model
# Null model includes only the intercept and the random effect (grouping variable)

null_model = smf.mixedlm(
    'excess_deaths_per_100k ~ 1',  # Only intercept
    county_data, 
    groups=county_data['state'],
    re_formula='~1'  # Random intercept
)
null_result = null_model.fit(reml=False)

# Display the null model results
print(null_result.summary())

# Extract 2LL, AIC, and BIC
two_ll = -2 * null_result.llf
aic = null_result.aic
bic = null_result.bic

# Print the results
print(f"2LL: {two_ll}")
print(f"AIC: {aic}")
print(f"BIC: {bic}")

               Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: excess_deaths_per_100k
No. Observations: 1142    Method:             ML                    
No. Groups:       44      Scale:              59.7142               
Min. group size:  5       Log-Likelihood:     -3996.5293            
Max. group size:  80      Converged:          Yes                   
Mean group size:  26.0                                              
-----------------------------------------------------------------------
              Coef.     Std.Err.      z       P>|z|    [0.025    0.975]
-----------------------------------------------------------------------
Intercept     13.388       0.653    20.518    0.000    12.109    14.667
Group Var     15.194       0.531                                       

2LL: 7993.058578882613
AIC: 7999.058578882613
BIC: 8014.180188053261


B. full model

In [345]:
county_data

,region_code,state,region,type,observed_deaths,expected_deaths,excess_deaths,excess_deaths_per_100k,peak,Classification,...,GRDP_CAGR,Income,r_older,Education,Unemployment,r_white,r_black,r_AmericanIndianandAlaskaNative,r_Asian,r_NativeHawaiianandOtherPacificIslander
0,US55123,Wisconsin,"Vernon County, WI","Pop-Growth, Econ-Decline",30.61,23.83,6.78,21.93,7,4,...,-0.36,60.04,19.53,23.51,2.70,96.67,0.24,0.15,0.34,0.04
1,US48209,Texas,"Hays County, TX",Growing,130.19,107.67,22.53,8.67,3,2,...,5.16,79.34,10.99,38.67,3.00,82.66,4.13,0.61,1.47,0.05
2,US1005,Alabama,"Barbour County, AL",Shrinking,30.00,29.69,0.31,1.24,8,5,...,-1.31,38.65,19.42,11.15,4.10,46.30,47.67,0.35,0.49,0.00
3,US48203,Texas,"Harrison County, TX","Pop-Growth, Econ-Decline",69.47,55.07,14.41,20.76,11,3,...,-2.22,56.65,16.90,21.12,4.50,71.58,20.71,0.34,0.81,0.03
4,US48091,Texas,"Comal County, TX",Growing,140.81,125.69,15.11,8.75,8,2,...,4.95,93.49,18.08,39.60,3.50,87.10,2.17,0.25,1.17,0.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,US1049,Alabama,"DeKalb County, AL",Growing,86.11,72.10,14.01,19.52,7,5,...,0.36,45.06,17.26,14.43,2.30,84.77,1.53,1.19,0.11,0.13
1138,US47167,Tennessee,"Tipton County, TN",Growing,65.03,50.98,14.04,22.91,8,2,...,1.27,62.81,14.46,18.68,3.60,76.97,18.54,0.25,0.70,0.00
1139,US55043,Wisconsin,"Grant County, WI",Growing,46.64,43.56,3.08,5.94,9,5,...,1.51,57.86,17.25,23.86,2.70,96.16,1.56,0.14,0.97,0.00
1140,US18113,Indiana,"Noble County, IN",Growing,44.25,44.68,-0.43,-0.91,7,5,...,1.29,65.98,16.04,16.31,2.80,94.85,0.63,0.05,0.69,0.07


In [347]:
county_data.columns

Index(['region_code', 'state', 'region', 'type', 'observed_deaths',
       'expected_deaths', 'excess_deaths', 'excess_deaths_per_100k', 'peak',
       'Classification', 'total_Area (Land, in square meters)',
       'total_Area (Land, in square kilo meters)', 'pop', 'PopDensity',
       'POP_CAGR', 'GRDP_CAGR', 'Income', 'r_older', 'Education',
       'Unemployment', 'r_white', 'r_black', 'r_AmericanIndianandAlaskaNative',
       'r_Asian', 'r_NativeHawaiianandOtherPacificIslander'],
      dtype='object')

In [351]:
# Fit a full mixed-effects model

model = smf.mixedlm(
    'excess_deaths_per_100k ~ POP_CAGR + GRDP_CAGR + Income + r_older + Education + Unemployment + r_white + r_black + r_Asian + r_NativeHawaiianandOtherPacificIslander + pop + PopDensity',  
    county_data, 
    groups=county_data['state'],
    re_formula='~1'
)

result = model.fit(reml=False)

# Display the results
print(result.summary())

# Extract 2LL, AIC, and BIC
two_ll = -2 * result.llf
aic = result.aic
bic = result.bic

# Print the results
print(f"2LL: {two_ll}")
print(f"AIC: {aic}")
print(f"BIC: {bic}")

                      Mixed Linear Model Regression Results
Model:                  MixedLM     Dependent Variable:     excess_deaths_per_100k
No. Observations:       1142        Method:                 ML                    
No. Groups:             44          Scale:                  43.4948               
Min. group size:        5           Log-Likelihood:         -3808.2599            
Max. group size:        80          Converged:              Yes                   
Mean group size:        26.0                                                      
----------------------------------------------------------------------------------
                                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------------------
Intercept                               21.628    4.699  4.603 0.000 12.419 30.837
POP_CAGR                                -0.795    0.383 -2.076 0.038 -1.546 -0.044
GRDP_CAGR                  

#### (2) Number of peaks

A. null model

In [316]:
# Fit a null mixed-effects model
# Null model includes only the intercept and the random effect (grouping variable)

null_model = smf.mixedlm(
    'peak ~ 1',  # Only intercept
    county_data, 
    groups=county_data['state'],
    re_formula='~1'  # Random intercept
)
null_result = null_model.fit(reml=False)

# Display the null model results
print(null_result.summary())

# Extract 2LL, AIC, and BIC
two_ll = -2 * null_result.llf
aic = null_result.aic
bic = null_result.bic

# Print the results
print(f"2LL: {two_ll}")
print(f"AIC: {aic}")
print(f"BIC: {bic}")

         Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: peak      
No. Observations: 1142    Method:             ML        
No. Groups:       44      Scale:              5.9828    
Min. group size:  5       Log-Likelihood:     -2676.8785
Max. group size:  80      Converged:          Yes       
Mean group size:  26.0                                  
---------------------------------------------------------
           Coef.  Std.Err.    z     P>|z|  [0.025  0.975]
---------------------------------------------------------
Intercept  6.678     0.180  37.147  0.000   6.326   7.030
Group Var  1.079     0.125                               

2LL: 5353.757046176354
AIC: 5359.757046176354
BIC: 5374.878655347003


B. full model

In [314]:
# Fit a full mixed-effects model

model = smf.mixedlm(
    'peak ~ POP_CAGR + GRDP_CAGR + Income + r_older + Education + Unemployment + r_white + r_black + r_Asian + r_NativeHawaiianandOtherPacificIslander + pop + PopDensity',  
    county_data, 
    groups=county_data['state'],
    re_formula='~1'
)
result = model.fit(reml=False)

# Display the results
print(result.summary())

# Extract 2LL, AIC, and BIC
two_ll = -2 * result.llf
aic = result.aic
bic = result.bic

# Print the results
print(f"2LL: {two_ll}")
print(f"AIC: {aic}")
print(f"BIC: {bic}")

                      Mixed Linear Model Regression Results
Model:                      MixedLM         Dependent Variable:         peak      
No. Observations:           1142            Method:                     ML        
No. Groups:                 44              Scale:                      2.6850    
Min. group size:            5               Log-Likelihood:             -2202.3180
Max. group size:            80              Converged:                  Yes       
Mean group size:            26.0                                                  
----------------------------------------------------------------------------------
                                        Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------------------------
Intercept                                6.487    1.071  6.056 0.000  4.388  8.587
POP_CAGR                                -0.608    0.086 -7.090 0.000 -0.776 -0.440
GRDP_CAGR                  